### Objective: Ingest CSVs into database
- create udf to load all csv files into targetted database

In [1]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType,
    ShortType, TimestampType
)
from pyspark.sql.functions import length, col, max
import os
import gc

In [2]:
spark = SparkSession.builder \
    .appName("AWS_Crawl_App") \
    .config(
        "spark.jars",
        "/home/jovyan/work/jars/postgresql-42.7.3.jar") \
    .getOrCreate()

jdbc_url = "jdbc:postgresql://postgres_db:5432/mydb"

In [3]:
spark

### Connection Properties

In [4]:
## Check config of postgres
POSTGRES_HOST = "postgres" # Docker service/container name from docker compose (NOT localhost)
POSTGRES_PORT = "5432" # Internal Docker port
POSTGRES_DB   = "aws_common" # Database name from PGADMIN
#POSTGRES_DB   = "postgres" # Database name from PGADMIN

# ── Assemble the URL ─────────────────────────────────────
jdbc_url = f"jdbc:postgresql://{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

connection_properties = {
    "user": "airflow",
    "password": "airflow",
    "driver": "org.postgresql.Driver"
}

### Database Parameters

In [5]:
TABLE_NAME= '"DBO"."STG_AWS_CC_RAW"'
POSTGRES_USER= 'airflow'
POSTGRES_PASS= 'airflow'

### Schema Inference

In [6]:
# ## Schema configuration on columns
# aws_crawl_schema = StructType([
#     StructField("ingest_id",                    StringType(),       False),
#     StructField("crawl_id",                     StringType(),       False),
#     StructField("url",                          StringType(),       False),
#     StructField("url_surtkey",                  StringType(),       True),
#     StructField("url_host_registered_domain",   StringType(),       True),
#     StructField("url_host_tld",                 StringType(),       True),
#     StructField("fetch_date",                   TimestampType(),    True),
#     StructField("fetch_status",                 ShortType(),        True),
#     StructField("content_mime_detected",        StringType(),       True),
#     StructField("content_languages",            StringType(),       True),
#     StructField("content_charset",              StringType(),       True),
#     StructField("warc_filename",                StringType(),       True),
#     StructField("warc_record_offset",           StringType(),       True),
#     StructField("warc_record_length",           StringType(),       True),
#     StructField("ingested_at",                  TimestampType(),    True),
# ])


## Schema configuration on columns
aws_crawl_schema = StructType([
    StructField("ingest_id",                    StringType(),       False),
    StructField("crawl_id",                     StringType(),       False),
    StructField("url",                          StringType(),       False),
    StructField("url_surtkey",                  StringType(),       True),
    StructField("url_host_registered_domain",   StringType(),       True),
    StructField("url_host_tld",                 StringType(),       True),
    StructField("fetch_date",                   StringType(),    True),
    StructField("fetch_status",                 ShortType(),        True),
    StructField("content_mime_detected",        StringType(),       True),
    StructField("content_languages",            StringType(),       True),
    StructField("content_charset",              StringType(),       True),
    StructField("warc_filename",                StringType(),       True),
    StructField("warc_record_offset",           StringType(),       True),
    StructField("warc_record_length",           StringType(),       True),
    StructField("ingested_at",                  TimestampType(),    True),
])


### NOTE: Update schema data type: 04092026

In [7]:
# ## NOTE: 
# aws_crawl_schema = StructType([
#     StructField("ingest_id",                    StringType(),   False),
#     StructField("crawl_id",                     StringType(),   False),
#     StructField("url",                          StringType(),   False),
#     StructField("url_surtkey",                  StringType(),   True),
#     StructField("url_host_registered_domain",   StringType(),   True),
#     StructField("url_host_tld",                 StringType(),   True),
#     StructField("fetch_date",                   TimestampType(),True),
#     StructField("fetch_status",                 ShortType(),    True),
#     StructField("content_mime_detected",        StringType(),   True),
#     StructField("content_languages",            StringType(),   True),
#     StructField("content_charset",              StringType(),   True),
#     StructField("warc_filename",                StringType(),   True),
#     StructField("warc_record_offset",           LongType(),     True),
#     StructField("warc_record_length",           LongType(),     True),
#     StructField("ingested_at",                  TimestampType(),True),
# ])

### File Path for `CSV`

In [8]:
csv_files_path= 'data/repartition_data/'
csv_files= os.listdir(csv_files_path)
csv_files[:5]

['part-00013-e9bb82ee-ab4f-49e3-b7f3-e0c8290aadff-c000.csv',
 '.part-00005-e9bb82ee-ab4f-49e3-b7f3-e0c8290aadff-c000.csv.crc',
 'part-00004-e9bb82ee-ab4f-49e3-b7f3-e0c8290aadff-c000.csv',
 '.part-00004-e9bb82ee-ab4f-49e3-b7f3-e0c8290aadff-c000.csv.crc',
 'part-00008-e9bb82ee-ab4f-49e3-b7f3-e0c8290aadff-c000.csv']

### UDF: to load csv on Target database

In [9]:
def ingest_in_db(
    spark_inst,
    file_path_: str, 
    file_ext: str,
    schema: str, 
    table_name: str, 
    jdbc_url: str, 
    conn_properties):
    '''
    Ingest into target table

    -------
    params:
        file_path (str): Path to filenames
        schema (str): Schema of Incoming files
        table_name (str): Target Table Name
        jdbc_url (str): 
        conn_properties (dict): Connection properties to postgres database

    '''

    FORMAT= 'jdbc'

    try:
        df_new= spark_inst.read\
            .format(file_ext)\
            .schema(schema)\
            .option("header", "false")\
            .load(file_path_)
        
        # Write the dataframe into the database with target table name
        df_new.write \
            .format(FORMAT) \
            .mode("append") \
            .option("url", jdbc_url) \
            .option("dbtable", TABLE_NAME) \
            .options(**conn_properties)\
            .save()


    except Exception as e:
        print(f"ERROR ingesting: {e}")

In [10]:
## ingest all files iteratively
file_path= 'data/repartition_data/'
csv_files= [file_ for file_ in os.listdir(csv_files_path) if file_.endswith('.csv')]
print(f"Total CSV repartition files: {len(csv_files)}") 
print(f"Sample File_names:\n {csv_files[:3]}")

Total CSV repartition files: 20
Sample File_names:
 ['part-00013-e9bb82ee-ab4f-49e3-b7f3-e0c8290aadff-c000.csv', 'part-00004-e9bb82ee-ab4f-49e3-b7f3-e0c8290aadff-c000.csv', 'part-00008-e9bb82ee-ab4f-49e3-b7f3-e0c8290aadff-c000.csv']


### Ingest individual files

ingest_in_db(
    spark_,
    file_path: str, 
    file_ext: str,
    schema: str, 
    table_name: str, 
    jdbc_url: str, 
    conn_properties)

In [11]:
%%time
TABLE_NAME= '"DBO"."STG_AWS_CC_RAW"'

file_ext= 'csv'
schema= aws_crawl_schema
FORMAT= 'jdbc'

try:
    ## Iterate over the csv files from list
    for file_num in range(0,len(csv_files)):
    
        ## Get File from path
        file_path_ = file_path + csv_files[file_num]
        
        df_new= spark.read\
            .format(file_ext)\
            .schema(schema)\
            .option("header", "false")\
            .load(file_path_)

        # df_new= spark.read\
        #     .format(file_ext)\
        #     .option("header", "false")\
        #     .load(file_path_)
        
        # Write the dataframe into the database with target table name
        df_new.write \
            .format(FORMAT) \
            .mode("append") \
            .option("url", jdbc_url) \
            .option("dbtable", TABLE_NAME) \
            .options(**connection_properties)\
            .save()
    
        del df_new
        gc.collect()

except Exception as e:
    print(f"Error Ingesting files: {e}")

# finally:
#     pass

# df_new.show(5)

Error Ingesting files: An error occurred while calling o372.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 6 in stage 17.0 failed 1 times, most recent failure: Lost task 6.0 in stage 17.0 (TID 142) (cf1a93788225 executor driver): java.sql.BatchUpdateException: Batch entry 335 INSERT INTO "DBO"."STG_AWS_CC_RAW" ("ingest_id","crawl_id","url","url_surtkey","url_host_registered_domain","url_host_tld","fetch_date","fetch_status","content_mime_detected","content_languages","content_charset","warc_filename","warc_record_offset","warc_record_length","ingested_at") VALUES (('ca'),('dewis'),('gordon)/2012/01/17/snowflakes?share=pinterest 20210116143151 {"url": "https://gordon.dewis.ca/2012/01/17/snowflakes/?share=pinterest"'),('"mime": "text/html"'),('"mime-detected": "text/html"'),('"status": "302"'),('"digest": "3I42H3S6NNFQ2MSVX7XZKYAYSCX5QBYJ"'),(NULL),('"offset": "10845295"'),('"filename": "crawl-data/CC-MAIN-2021-04/segments/1610703506697.14/crawldiagnostic

In [ ]:
# TABLE_NAME= '"DBO"."STG_AWS_CC_RAW"'

# for file_num in range(0, 3):
#     file_name = file_path + csv_files[file_num]
    
#     ingest_in_db(spark, file_name, 'str', aws_crawl_schema, TABLE_NAME, jdbc_url, connection_properties)    